## Sentiment Analysis

### Classify documents based on their sentiment using ML

We will download the movie review dataset from IMDB.

In [76]:
import tarfile
import pandas as pd
import numpy as np
import pyprind
import re
import os
import sys
import nltk
# import ssl

# try:
#     _create_unverified_https_context = ssl._create_unverified_context
# except AttributeError:
#     pass
# else:
#     ssl._create_default_https_context = _create_unverified_https_context

# # Now your download will bypass the SSL handshake error
# nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer, HashingVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, SGDClassifier

In [ ]:
# Unzip tar file 
with tarfile.open("data/aclImdb_v1.tar","r") as tar:
    tar.extractall(path="./data/")

In [10]:
basepath = 'data/aclImdb'

In [ ]:
## Reading reviews from both test and train dataset and appending them into one dataset and creating a label.
labels = {'pos': 1, 'neg': 0}
pbar = pyprind.ProgBar(50000, stream = sys.stdout)
df = pd.DataFrame()

for s in ('test', 'train'):
    for l in ('pos','neg'):
        path = os.path.join(basepath, s, l)
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file), 'r', encoding = 'utf-8') as infile:
                txt = infile.read()
            df = df.append([[txt, labels[l]]],
                           ignore_index = True)
            pbar.update()

df.columns = ['review', 'sentiment']

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:04:05


In [ ]:
## Reshuffle the index so we have a mix of both positive and negative review.
np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))
# save the data.
df.to_csv('data/movie_data.csv', index = False, encoding='utf-8')

In [23]:
df = pd.read_csv('data/movie_data.csv', encoding='utf-8')
df.head()

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0
3,hi for all the people who have seen this wonde...,1
4,"I recently bought the DVD, forgetting just how...",0


## Bag-of-words model

Bag of word model allow us to represent text as numerical features vectors.

The idea behind bag-of-words model is :-
1. we create a vocabulary of unique tokens - for example, words - from the entire set of documents.
2. we construct  a feature vector from each document that contains the count of how often each word occurs in the particular document.

Since the unique word in a document is a small subset of bag-of-words vocabulary, most feature vector will be 0, hence we call them sparse.

### Transform words into feature vectors

We can use CountVectorizer that take an array of text and create bag-of-words model for us.

In [ ]:
## An Example
count = CountVectorizer()
docs = np.array(['The sun is shining',
        'The weather is sweet',
        'The sun is shining, the weather is sweet, and one and one is two'])
bag = count.fit_transform(docs) # construct vocab of bag-of-world model and transformed them into feature vector

In [ ]:
# bag-of-words vocabulary
count.vocabulary_

{'the': 6,
 'sun': 4,
 'is': 1,
 'shining': 3,
 'weather': 8,
 'sweet': 5,
 'and': 0,
 'one': 2,
 'two': 7}

The values for each word is their index. which means that 'and' has index 0, 'is' has index 1 and so on. The indexes are usually assigned alphabetically. So feature vector the value in index 0 will represent how many occurance of word 'and' has in a particulat text.

In [ ]:
## Sparse feature vectors
print(bag.toarray())

[[0 1 0 1 1 0 1 0 0]
 [0 1 0 0 0 1 1 0 1]
 [2 3 2 1 1 1 2 1 1]]


The values in the feature vectors are also called ***raw term frequencies***. 

***tf(t,d)*** - number of times a word t occur in the document d.

#### N-grams model

The contigous sequence of items in NLP - words, letters or symbols are also n-grams. The choice of number n depends on the kind of application. For e.g. in spam filtering, n-grams of size 3 or 4 yield good results. 

For a document "The sun is shining", 1-gram, 2-gram representation would be:-
* 1-gram: "The", "sun", "is", "shining".
* 2-gram: "The sun", "sun is", "is shining".


## term frequency-inverse document frequency (tf-idf)

During analysis, we often come across words that appear in all the documents from all classes. These frequently used words don't contain any valuable of differentiating information. We can use tf-idf to downweight such words in the feature vectors. The tf-idf can be defined as :-

$$tf-idf(t,d) = tf(t,d) * idf(t,d)$$
where idf(t,d) is given as 

$$idf(t,d) = log \frac{n_d}{1 + df(d,t)}$$
where $n_d$ is the total number of documents and df(t,d) is no. of documents contain the word t. The log is used to ensure low frequency words are not given too much weight. 1 is added just to make sure we have non-zero value of idf.


In [32]:
tfidf = TfidfTransformer(use_idf= True,
                         norm='l2',
                         smooth_idf=True)
np.set_printoptions(precision=2)
print(tfidf.fit_transform(count.fit_transform(docs)).toarray())


[[0.   0.43 0.   0.56 0.56 0.   0.43 0.   0.  ]
 [0.   0.43 0.   0.   0.   0.56 0.43 0.   0.56]
 [0.5  0.45 0.5  0.19 0.19 0.19 0.3  0.25 0.19]]


We can see that word "is" that has highest frequency in 3rd document is downweighted and now has tf-idf of 0.45 as it is also contained in other documents hence unlikely to have any useful information.

However, the scikit-learn has a different formula for idf.
$$idf(t,d) = log \frac{1 + n_d}{1 + df(d,t)}$$
and tf-idf would be 
$$tf-idf(t,d) = tf(t,d) * (idf(t,d) + 1)$$
The +1 is for setting smooth-idf = True, which is helpful for assigning 0 weight to terms occur in all documents.

It is also common to normalize term frequencies before calculating tf-idf. However, sklearn directly normalize tf-idfs. For L2 norm,
$$v_{norm} = \frac{v}{||v||_2}$$

## Cleaning text data

Text data may contain a lot of punctuation, HTML tags, other non letter characters. Some punctuations can be useful in some context.

In [34]:
df.loc[0,'review'][-50:]

'is seven.<br /><br />Title (Brazil): Not Available'

To remove these html, punctuation tags, we can use regex library. Word capitalization can be useful in some context but here we will convert everything to lower case.

In [39]:
def preprocessor(text):
    # For removing html/xml tags
    # map < bracket and Matches zero or more characters of any kind, except for a closing angle bracket using [^>]*
    # followed by a closing bracket >. which will remove html tags like <div>, etc.
    text = re.sub('<[^>]*>','',text) 

    # For removing emoticons by identifying eyes, nose, and mouth
    # (?::|;|=) - maps eyes as :, ;, =
    # (?:-) - maps nose as hyphen - exactly zero or one time because of the ? quantifier.
    # (?:\)|\(|D|P) - maps mouth as \, D, P
    # he ?: inside the parentheses creates non-capturing groups. 
    # This ensures re.findall returns the full matched emoticon string rather than just pieces of it.
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)

    # For removing all spaces, punctuation, symbols
    # [\W]+ - [] is for character class. \W means all non-word items.
    # + - Matches one or more of these non-word characters in a row to speed up substitution
    text = (re.sub('[\W]+', ' ', text.lower()) + 
            ' '.join(emoticons).replace('-', '')) # removing nose emoticons for consistency
    
    return text


In [ ]:
#Testing the function
preprocessor(df.loc[0,'review'][-50:])

'is seven title brazil not available'

In [41]:
#Cleaning the whole data
df['review'] = df['review'].apply(preprocessor)

### Processing documents into tokens

After the data preparation, we need to think about how to split text corpora into individual elements which is called ***tokenization***. 

* One way to tokenize is to split the text into individual terms.

In [42]:
def tokenizer(text):
    return(text.split())

tokenizer('runners like running and thus they run')

['runners', 'like', 'running', 'and', 'thus', 'they', 'run']

* Another technique of tokenization, is called word ***stemming***, which is a process of transforming a word to its root form. For e.g. 'running' becomes 'run'.

In [47]:
porter = PorterStemmer()
def tokenizer_porter(text):
    return [porter.stem(word) for word in text.split()]

tokenizer_porter('runners like running and thus they run')

['runner', 'like', 'run', 'and', 'thu', 'they', 'run']

### Stemming Algorithms

The porter stemming algorithm is the oldest one. There are newer and faster algorithm like Snowball stemmer, Lancaster stemmer. The new algorithm are more aggressive than porter algorithm as they produce shorter and obscure words.

### Lemmatization

While stemming create non-real words, lemmatization aim to obtain the canonical (grammatically correct) forms of individual words - which are called lemmas. However, lemmatization is computationally more difficult and expensive compared to stemming and also has little effect on performance fot text classification.

### Stop words

Stop words are the words that are extremly common and probably bear no useful information. Removing them can be useful if we are working with raw frequency rather than tf-idf, which already downweight the frequently occuring words.

In [ ]:
# removing stop words
stop = stopwords.words('english')
[w for w in tokenizer_porter('a runners like running and thus they run') if w not in stop]

['runner', 'like', 'run', 'thu', 'run']

### Training a logistic regression model for document classification

In [55]:
X_train = df.loc[:25000, 'review'].values
Y_train = df.loc[:25000, 'sentiment'].values
X_test = df.loc[25000:, 'review'].values
Y_test = df.loc[25000:, 'sentiment'].values

In [58]:
tfidf = TfidfVectorizer(strip_accents=None,
                        lowercase=False,
                        preprocessor=None)

small_param_grid = [
    {
        'vect__ngram_range': [(1,1)], # means 1-gram model, (1,2)  will consider both 1 and 2-gram model
        'vect__stop_words': [None],
        'vect__tokenizer': [tokenizer, tokenizer_porter],
        'clf__penalty': ['l2'],
        'clf__C': [1.0, 10.0]
    },
    {
        'vect__ngram_range': [(1,1)],
        'vect__stop_words': [stop, None],
        'vect__tokenizer': [tokenizer],
        'vect__use_idf': [False],
        'vect__norm': [None],
        'clf__penalty': ['l2'],
        'clf__C': [1.0, 10.0]
    }
]

lr_tfidf = Pipeline([
    ('vect', tfidf),
    ('clf', LogisticRegression(solver='liblinear'))
])

gs_lr_tfidf = GridSearchCV(lr_tfidf, small_param_grid,
                           scoring='accuracy', cv=5,
                           verbose=2, n_jobs = 1)
gs_lr_tfidf.fit(X_train, Y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x7faabcfc16a8>; total time=   7.2s
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x7faabcfc16a8>; total time=   6.4s
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x7faabcfc16a8>; total time=   5.0s
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x7faabcfc16a8>; total time=   4.9s
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer at 0x7faabcfc16a8>; total time=   5.1s
[CV] END clf__C=1.0, clf__penalty=l2, vect__ngram_range=(1, 1), vect__stop_words=None, vect__tokenizer=<function tokenizer_porter 

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('vect',
                                        TfidfVectorizer(lowercase=False)),
                                       ('clf',
                                        LogisticRegression(solver='liblinear'))]),
             n_jobs=1,
             param_grid=[{'clf__C': [1.0, 10.0], 'clf__penalty': ['l2'],
                          'vect__ngram_range': [(1, 1)],
                          'vect__stop_words': [None],
                          'vect__tokenizer': [<function tokenizer at 0x7faabcfc16a8>,
                                              <function tokenizer_porter at 0x7faac8067d08>...
                          'vect__stop_words': [['a', 'about', 'above', 'after',
                                                'again', 'against', 'ain',
                                                'all', 'am', 'an', 'and', 'any',
                                                'are', 'aren', "aren't", 'as',
                         

The idea of the grid above is to test if stemming, idf has an effect or not over raw frequency and no stemming.

In [59]:
print(gs_lr_tfidf.best_params_)

{'clf__C': 10.0, 'clf__penalty': 'l2', 'vect__ngram_range': (1, 1), 'vect__stop_words': None, 'vect__tokenizer': <function tokenizer at 0x7faabcfc16a8>}


We got the best results with tf-idf with no stop words, and stemming combined with LogisticRegression that use L2 regularization with C as 10.

In [60]:
print(f'CV accuracy is : {gs_lr_tfidf.best_score_: .3f}')

CV accuracy is :  0.897


In [61]:
clf = gs_lr_tfidf.best_estimator_
print(f'Test accuracy : {clf.score(X_test, Y_test): .3f}')

Test accuracy :  0.899


### Online learning Out-of-core Learning

For a big dataset, training such large models and creating feature vectors can be time consuming. Hence, we will use stochastic gradient descent to train data on small batches.

In [62]:
def tokenizer(text):
    text = re.sub('<[^>]*>','',text)
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text.lower())
    text = (re.sub('[\W]+', ' ', text.lower()) + 
            ' '.join(emoticons).replace('-', ''))
    tokenized = [w for w in text.split() if w not in stop]
    return tokenized

In [63]:
# Generator function that reads in and return one document at a time.
def stream_docs(path):
    with open(path, 'r', encoding = 'utf-8') as csv:
        next(csv) #skip header
        for line in csv:
            text, label = line[:-3], int(line[-2])
            yield text, label

In [65]:
next(stream_docs(path = 'data/movie_data.csv'))

('"In 1974, the teenager Martha Moxley (Maggie Grace) moves to the high-class area of Belle Haven, Greenwich, Connecticut. On the Mischief Night, eve of Halloween, she was murdered in the backyard of her house and her murder remained unsolved. Twenty-two years later, the writer Mark Fuhrman (Christopher Meloni), who is a former LA detective that has fallen in disgrace for perjury in O.J. Simpson trial and moved to Idaho, decides to investigate the case with his partner Stephen Weeks (Andrew Mitchell) with the purpose of writing a book. The locals squirm and do not welcome them, but with the support of the retired detective Steve Carroll (Robert Forster) that was in charge of the investigation in the 70\'s, they discover the criminal and a net of power and money to cover the murder.<br /><br />""Murder in Greenwich"" is a good TV movie, with the true story of a murder of a fifteen years old girl that was committed by a wealthy teenager whose mother was a Kennedy. The powerful and rich f

In [71]:
def get_minibatch(doc_stream, size):
    docs, y = [], []
    try:
        for _ in range(size):
            text, label = next(doc_stream)
            docs.append(text)
            y.append(label)
    except StopIteration:
        return None, None
    return docs, y

We can't use countVectorizer as it holds complete vocabulary in the memory. tf-idf also keep all the feature vectors to calculate inverse frequencies. Another useful vectorizer that can be used for online learning is ***HashingVectorizer***.

### Hashing Vectorizer

It is a stateless text-to-matrix converter that applies the "hashing trick".

For hashing vectorizer, we tell upfront how wide the sparse array is going to be like, for e.g. 100. which is called as "no. of features" The length of the array will be 100. And then each token will be given a slot in the array based on a hashing trick.

* Hashing function - It is a function that takes an input token and output a randon but consistent big number. for e.g hash("is") -> 1687345008.
Then we take a mod of this random number with "feature count". For e.g.
$1687345008\ \%\ 100$. to figure out what the remainder is. 

This remainder will decide the slot that token will get in the matrix. 

For e.g. for a text "I am happy" and no. of features as 10.

we will calculate hash functions and take mod with 10. let's assume we get following remainder. \
hash("i") = 6 \
hash("am") = 8 \
hash("happy") = 1 \
The array representation of text will be 
[0,1,0,0,0,0,1,0,1,0].

* no. of features:- While deciding no. of features, we need to keep in mind that a small value might result in ***hash collision*** which means that 2 tokens might get the same slot. Hence we should choose a number big enough (depends on the document size as well).

HashingVectorizer requires less memory.

In [70]:
vect = HashingVectorizer(decode_error='ignore',
                         n_features=2**21,
                         tokenizer=tokenizer)
clf = SGDClassifier(loss='log', random_state=1)
doc_stream = stream_docs(path='data/movie_data.csv')

Even though large n_features reduce the risk of hash collison, it has also increased the no. of coefficient in the logistic regression model.

In [72]:
pbar = pyprind.ProgBar(45)
classes = np.array([0,1])
for _ in range(45): # 45 iterations with a mini batch of size 1000 each.
    X_train, y_train = get_minibatch(doc_stream, size = 1000)
    if not X_train:
        break
    X_train = vect.transform(X_train)
    clf.partial_fit(X_train, y_train, classes = classes)
    pbar.update()

We can now test the accuracy on the last 5000 documents.

In [73]:
X_test, y_test = get_minibatch(doc_stream, size = 5000)
X_test = vect.transform(X_test)
print(f"Accuracy: {clf.score(X_test, y_test):.3f}")

Accuracy: 0.868


Test accuracy is 86.8%. Now we can use the last 5000 for updating the model.

In [74]:
clf = clf.partial_fit(X_test, y_test)

## Topic Modeling with latent Dirichlet Allocation

***Topic Modeling*** describes the broad task of assigning topic to unlabeled text documents. For e.g. sports, finance, politics, etc. Topic modeling is considered a clustering task.

### Decomposing text with LDA

LDA is a generative probabilistic model that tries to fins groups of words that appear frequently together across different documents. These frequently appearing words represent our topics, assuming that each document is a mixture of different words.

The input of LDA is a bag-of-words model.

Given a bag-of-words matrix as input, LDA decomposes it into two new matrix:-
* A document-to-topic matrix.
* A word-to-topic matrix.

LDA decomposes the bag-of-words matrix in such a way that if we multiply these two matrix, we will be able to reproduce the input with lowest error possible. 

* One downside of LDA is that we have to define number of topic beforehand - which is like a hyperparameter in LDA.

LDA uses expectation-maximization algorithm to update the parameters estimates iteratively.

In [75]:
df = pd.read_csv('data/movie_data.csv', encoding='utf-8')

In [78]:
# Now we will create bag-of-words matrix using CountVectorizer
count = CountVectorizer(stop_words='english',
                        max_df=.1, # maximum document frequency is 10% to exclude words that occur more frequently.
                        max_features=5000) #No. of words of to be considered is top 5000.
X = count.fit_transform(df['review'].values)

In [79]:
lda = LatentDirichletAllocation(n_components=10,
                                random_state=123,
                                learning_method='batch') # we are using all the available training data. 
#Another option is online, which will use mini-batch learning.
X_topics = lda.fit_transform(X)

In [ ]:
lda.components_.shape # 10 topics * 5000 words matrix. This contains the word importance.

(10, 5000)

In [84]:
n_top_words = 5
feature_names = count.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_):
    print(f'Topic {topic_idx + 1}:')
    print(' '.join([feature_names[i]
                    for i in topic.argsort()[:-n_top_words - 1: -1]]))

Topic 1:
worst minutes awful script stupid
Topic 2:
family mother father children girl
Topic 3:
american war dvd music tv
Topic 4:
human audience cinema art sense
Topic 5:
police guy car dead murder
Topic 6:
horror house sex girl woman
Topic 7:
role performance comedy actor performances
Topic 8:
series episode war episodes tv
Topic 9:
book version original read novel
Topic 10:
action fight guy guys cool


Based on the top 5 words, we can classify movies in 10 different genre. To make sure that categories make sense, we can print a few of them for topic 6 which is horror.

In [85]:
X_topics.shape

(50000, 10)

In [90]:
horror = X_topics[:, 5].argsort()[::-1] # category 6
for iter_idx, movie_idx in enumerate(horror[:3]):
    print(f'\nHorror movie #{(iter_idx + 1)}:')
    print(df['review'][movie_idx][:300], '...')


Horror movie #1:
House of Dracula works from the same basic premise as House of Frankenstein from the year before; namely that Universal's three most famous monsters; Dracula, Frankenstein's Monster and The Wolf Man are appearing in the movie together. Naturally, the film is rather messy therefore, but the fact that ...

Horror movie #2:
Okay, what the hell kind of TRASH have I been watching now? "The Witches' Mountain" has got to be one of the most incoherent and insane Spanish exploitation flicks ever and yet, at the same time, it's also strangely compelling. There's absolutely nothing that makes sense here and I even doubt there  ...

Horror movie #3:
<br /><br />Horror movie time, Japanese style. Uzumaki/Spiral was a total freakfest from start to finish. A fun freakfest at that, but at times it was a tad too reliant on kitsch rather than the horror. The story is difficult to summarize succinctly: a carefree, normal teenage girl starts coming fac ...


In [ ]:
horror = X_topics[:, 5].argsort()[::1] # category 6


(50000,)

In [88]:
horror

array([25114, 48404, 31616, ..., 35613, 36082, 29185])